# OCR text detection practice

This section will introduce how to use PaddleOCR to complete the training and operation of the text detection DB algorithm, including:
1. Quickly call PaddleOCR package to experience text detection
2. Understand the principle of text detection DB algorithm
3. Master the text detection model construction process
4. Master the text detection model training process

Note: `paddleocr` refers to `PaddleOCR whl package`.


# 2. Detailed Implementation of DB Text Detection Algorithm


## 2.1 DB Text Detection Algorithm Principle


[DB](https://arxiv.org/pdf/1911.08947.pdf) is a segmentation-based text detection algorithm, which proposes a Differentiable Threshold Differentiable Binarization module (DB module) that uses a dynamic threshold to distinguish the text area from the background.


<center><img src="https://ai-studio-static-online.cdn.bcebos.com/5eabdb59916a4267a049e5440f5093a63b6bfac9010844fb971aad0607d455a1" width = "600"></center>
<center><br>Figure 2: The difference between DB model and other methods</br></center>
<br></br>

The process of the segmentation-based ordinary text detection algorithm is shown by the blue arrow in the above figure. After this method obtains the segmentation result, a fixed threshold is used to obtain the binarized segmentation map, and then heuristic algorithms such as pixel clustering are used to obtain Text area.

The flow of the DB algorithm is shown by the red arrow in the figure. The biggest difference is that DB has a threshold map, which uses the network to predict the threshold at each position of the picture, instead of using a fixed value to better separate the text background and the foreground. .

The DB algorithm has the following advantages:
1. The algorithm structure is simple, without tedious post-processing
2. Have good accuracy and performance on open source data


In the traditional image segmentation algorithm, after obtaining the probability map, the standard binarize method is used for processing. The pixels below the threshold are set to 0, and the pixels above the threshold are set to 1. The formula is as follows:
$$ B_{i,j}=\left\{
\begin{aligned}
1, if P_{i,j} >= t ,\\
0, otherwise.
\end{aligned}
\right.
$$
But the standard binarization method is not differentiable, making the network unable to train end-to-end. To solve this problem, the DB algorithm proposes Differentiable Binarization (DB). Differentiable binarization approximates the step function in standard binarization, using the following formula to replace:
$$
\hat{B} = \frac{1}{1 + e^{-k(P_{i,j}-T_{i,j})}}
$$
Among them, P is the probability map obtained above, T is the threshold map obtained above, and k is the gain factor. In the experiment, it is selected as 50 based on experience. The comparison chart of standard binarization and differentiable binarization is shown in **Figure 2(a)** below.

When using cross-entropy loss, the loss of positive and negative samples are $l_+$ and $l_-$ respectively:
$$
l_+ = -log(\frac{1}{1 + e^{-k(P_{i,j}-T_{i,j})}})
$$
$$
l_- = -log(1-\frac{1}{1 + e^{-k(P_{i,j}-T_{i,j})}})
$$
Taking the partial derivative of the input $x$ will give:
$$
\frac{\delta{l_+}}{\delta{x}} = -kf(x)e^{-kx}
$$
$$
\frac{\delta{l_-}}{\delta{x}} = -kf(x)
$$
It can be found that the enhancement factor will amplify the gradient of the error prediction, thereby optimizing the model to obtain better results. **Figure 2(b)**, the part of $x<0$ is the case where positive samples are predicted to be negative samples. It can be seen that the gain factor k amplifies the gradient; while **Figure 2(c)** The part where $x>0$ is a negative sample is predicted to be a positive sample, the gradient is also magnified.

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/29255d870bd74403af37c8f88cb10ebca0c3117282614774a3d607efc8be8c84" width = "600"></center>
<center><br>Figure 3: Schematic diagram of DB algorithm</br></center>
<br></br>



The overall structure of the DB algorithm is shown in the figure below:

<center><img src="https://ai-studio-static-online.cdn.bcebos.com/6e1f293e9a1f4c90b6c26919f16b95a4a85dcf7be73f4cc99c9dc5477bb956e6" width = "1000"></center>
<center><br>Figure 4: Schematic diagram of DB model network structure</br></center>
<br></br>

The input image is extracted through the network Backbone and FPN to extract features, and the extracted features are cascaded together to obtain a quarter-size feature of the original image. Then, the convolutional layer is used to obtain the text area prediction probability map and the threshold map respectively, and then through the DB The post-processing to get the text enclosing curve.


In [1]:
# For the first run, you need to open the comment on the next line and download the PaddleOCR code
#!git clone https://gitee.com/paddlepaddle/PaddleOCR
import os
# Modify the default directory where the code runs to /home/aistudio/PaddleOCR
os.chdir("PaddleOCR")

In [2]:
!pwd

/home/loc/Workspace/ocr-exp/PaddleOCR


## 2.2 DB Text Detection Model Construction


The DB text detection model can be divided into three parts:
- Backbone network, responsible for extracting image features
- FPN network, feature pyramid structure enhanced features
- Head network, calculate the probability map of the text area

In this section, PaddlePaddle is used to implement the above three network modules and complete the complete network construction.


**backbone network**

The Backbone part of the DB text detection network uses an image classification network. In the paper, ResNet50 is used. In this section of the experiment, in order to speed up the training, the MobileNetV3 large structure is used as the backbone.

In [4]:
import paddle

ModuleNotFoundError: No module named 'paddle'

In [3]:
#  https://github.com/PaddlePaddle/PaddleOCR/blob/release%2F2.4/ppocr/modeling/backbones/det_mobilenet_v3.py
from ppocr.modeling.backbones.det_mobilenet_v3 import MobileNetV3

ModuleNotFoundError: No module named 'paddle'